In [1]:
# Installation automatique de seaborn si besoin
try:
    import seaborn as sns
except ImportError:
    import sys
    !{sys.executable} -m pip install seaborn
    import seaborn as sns

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)


ImportError: 

IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!

Importing the numpy C-extensions failed. This error can happen for
many reasons, often due to issues with your setup or how NumPy was
installed.

We have compiled some common reasons and troubleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python 3.14 from "c:\Users\wizab\OneDrive\Documents\GitHub\projet-reseau\.venv\Scripts\python.exe"
  * The NumPy version is: "2.4.3"

and make sure that they are the versions you expect.

Please carefully study the information and documentation linked above.
This is unlikely to be a NumPy issue but will be caused by a bad install
or environment on your machine.

Original error was: DLL load failed while importing _multiarray_umath: La procédure spécifiée est introuvable.


# Web Application Firewall (WAF) intelligent basé sur le Machine Learning

## Introduction
Ce notebook présente la conception d'un WAF intelligent capable de détecter les requêtes web malicieuses à l'aide de techniques de machine learning.

### Contexte
La sécurité des applications web est un enjeu majeur. Un WAF permet de protéger les sites contre les attaques courantes (injections, scans, etc.).

### Objectifs
- Détecter automatiquement les requêtes malicieuses.
- Utiliser un jeu de données synthétique représentatif du trafic web.
- Comparer plusieurs modèles de machine learning.
- Proposer une solution intégrable dans une API Flask.

### Jeu de données
Le jeu de données utilisé est généré de façon synthétique pour simuler des requêtes normales et malicieuses. Il permet de tester la robustesse des modèles.

### Résumé du notebook
1. Exploration et visualisation des données
2. Nettoyage et préparation
3. Feature engineering
4. Entraînement et évaluation de modèles
5. Optimisation et interprétation
6. Discussion et perspectives
7. Export et utilisation du modèle

## 1. Importer les bibliothèques nécessaires
On commence par importer les bibliothèques Python courantes pour l'analyse de données et la visualisation.

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

# Affichage des graphiques dans le notebook
%matplotlib inline

ImportError: 

IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!

Importing the numpy C-extensions failed. This error can happen for
many reasons, often due to issues with your setup or how NumPy was
installed.

We have compiled some common reasons and troubleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python 3.14 from "c:\Users\wizab\OneDrive\Documents\GitHub\projet-reseau\.venv\Scripts\python.exe"
  * The NumPy version is: "2.4.3"

and make sure that they are the versions you expect.

Please carefully study the information and documentation linked above.
This is unlikely to be a NumPy issue but will be caused by a bad install
or environment on your machine.

Original error was: DLL load failed while importing _multiarray_umath: La procédure spécifiée est introuvable.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Affichage des graphiques dans le notebook
%matplotlib inline

In [ ]:
import warnings
import joblib
from sklearn import __version__ as sklearn_version

warnings.filterwarnings('ignore')

print('Versions utilisées :')
print('pandas', pd.__version__)
print('numpy', np.__version__)
print('matplotlib', plt.matplotlib.__version__)
print('seaborn', sns.__version__)
print('scikit-learn', sklearn_version)
print('joblib', joblib.__version__)


## 2. Charger et explorer les données
Charger un fichier de données (par exemple un CSV) et afficher les premières lignes pour explorer la structure des données.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Génération du dataset synthétique
num_requests = 100
start_time = datetime(2023, 10, 26, 8, 0, 0)
timestamps = [start_time + timedelta(seconds=i * np.random.randint(1, 10)) for i in range(num_requests)]
ip_addresses = np.random.choice([f'192.168.1.{i}' for i in range(1, 20)] + [f'10.0.0.{i}' for i in range(1, 10)], num_requests)
request_methods = np.random.choice(['GET', 'POST', 'PUT', 'DELETE'], num_requests)
normal_paths = ['/', '/index.html', '/login', '/products', '/api/data', '/dashboard']
malicious_paths = [
    '/etc/passwd', '/proc/self/cwd',
    '/login?user=%27%20OR%201=1--',
    '/admin/users.php?id=1%20UNION%20SELECT%20null,null,null,version()--',
    '/../../../../windows/system32/cmd.exe',
    '/phpmyadmin/index.php?pma_username=root',
    '/wp-admin/admin-ajax.php',
    '/shell.php', '/upload.php?file=evil.php'
]
request_paths = []
is_malicious = np.zeros(num_requests, dtype=bool)
for i in range(num_requests):
    if np.random.rand() < 0.15:
        request_paths.append(np.random.choice(malicious_paths))
        is_malicious[i] = True
    else:
        request_paths.append(np.random.choice(normal_paths))
status_codes = np.random.choice([200, 301, 400, 401, 403, 404, 500], num_requests)
user_agents = np.random.choice([
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/107.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/107.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 10) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/107.0.0.0 Mobile Safari/537.36',
    'curl/7.64.1', 'Python-requests/2.28.1',
    'Nikto/2.1.6',
    'sqlmap/1.6.10 (http://sqlmap.org)',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/107.0.0.0 Safari/537.36'
], num_requests)
data = {
    'timestamp': timestamps,
    'ip_address': ip_addresses,
    'request_method': request_methods,
    'request_path': request_paths,
    'status_code': status_codes,
    'user_agent': user_agents,
    'is_malicious': is_malicious
}
df = pd.DataFrame(data)

# Exploration rapide
print("Synthetic dataset created successfully!")
display(df.head())
df.info()
df.describe(include='all')

## 3. Nettoyer les données
Traiter les valeurs manquantes, convertir les types de données si nécessaire et supprimer les doublons.

In [ ]:
# Nettoyage des données
# Suppression des doublons
print(f"Avant suppression des doublons : {len(df)} lignes")
df = df.drop_duplicates()
print(f"Après suppression des doublons : {len(df)} lignes")

# Vérification des valeurs manquantes
print("Valeurs manquantes par colonne :")
print(df.isnull().sum())

# Conversion éventuelle des types (exemple : timestamp)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Vérification des types de données
df.dtypes

## 4. Analyser les données
Effectuer des analyses statistiques de base, comme la moyenne, la médiane et la distribution des variables.

In [ ]:
!pip install seaborn# Statistiques descriptives
df.describe()
# Distribution d'une variable (exemple)
# df['nom_colonne'].hist()

## 5. Visualiser les résultats
Créer des graphiques pour visualiser les tendances et les relations dans les données.

In [ ]:
# Exemple de visualisation
import matplotlib.pyplot as plt
import seaborn as sns
df.select_dtypes(include=[float, int]).hist(figsize=(12,8), bins=20)
plt.suptitle('Distribution des variables numériques')
plt.show()

## 6. Enregistrer les résultats
Enregistrer les résultats de l'analyse ou les données nettoyées dans un nouveau fichier.

## 7. Extraction de caractéristiques (feature engineering)
Pour la détection d'attaques web, il est important d'extraire des caractéristiques pertinentes des requêtes HTTP. Voici quelques exemples :
- Longueur de l'URL
- Nombre de paramètres
- Présence de caractères ou mots-clés suspects (ex : SELECT, UNION, <script>, ../, OR 1=1)
- Structure de la requête
- Fréquence de certains caractères spéciaux

In [ ]:
# Feature engineering automatisé
import re

def extract_features(df):
    # Longueur du chemin de la requête
    df['path_length'] = df['request_path'].apply(lambda x: len(str(x)))
    # Nombre de paramètres dans l'URL
    df['param_count'] = df['request_path'].apply(lambda x: str(x).count('&'))
    # Présence de mots-clés suspects
    keywords = ['select', 'union', 'script', '../', 'or 1=1', 'cmd.exe', 'passwd', 'shell', 'upload', 'phpmyadmin', 'admin']
    for kw in keywords:
        df[f'has_{kw.replace("/", "slash").replace(".", "dot").replace(" ", "_")}'] = df['request_path'].str.lower().apply(lambda x: int(kw in x))
    # Nombre de caractères spéciaux
    df['special_char_count'] = df['request_path'].apply(lambda x: len(re.findall(r'[;=(){}\[\]<>]', str(x))))
    # Méthode HTTP encodée
    df['method_GET'] = (df['request_method'] == 'GET').astype(int)
    df['method_POST'] = (df['request_method'] == 'POST').astype(int)
    df['method_PUT'] = (df['request_method'] == 'PUT').astype(int)
    df['method_DELETE'] = (df['request_method'] == 'DELETE').astype(int)
    # Code statut encodé
    for code in [200, 301, 400, 401, 403, 404, 500]:
        df[f'status_{code}'] = (df['status_code'] == code).astype(int)
    # User agent suspects
    df['user_agent_sqlmap'] = df['user_agent'].str.contains('sqlmap', case=False, na=False).astype(int)
    df['user_agent_nikto'] = df['user_agent'].str.contains('nikto', case=False, na=False).astype(int)
    df['user_agent_curl'] = df['user_agent'].str.contains('curl', case=False, na=False).astype(int)
    df['user_agent_python'] = df['user_agent'].str.contains('python', case=False, na=False).astype(int)
    return df

df = extract_features(df)
df.head()

## 8. Séparation des données et préparation pour le machine learning
On sépare les données en un ensemble d'entraînement et un ensemble de test, puis on prépare les variables explicatives (features) et la variable cible (label).

In [ ]:
# Séparation des données et préparation pour le machine learning
from sklearn.model_selection import train_test_split

# Sélection des features (on enlève les colonnes non pertinentes)
features = [col for col in df.columns if col not in [
    'timestamp', 'ip_address', 'request_method', 'request_path', 'status_code', 'user_agent', 'is_malicious'
]]
X = df[features]
y = df['is_malicious']

# Séparation train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Taille train : {X_train.shape}, Taille test : {X_test.shape}")
print(f"Répartition des classes dans y_train :\n{y_train.value_counts(normalize=True)}")

## 9. Entraînement et comparaison de plusieurs modèles de machine learning
On teste plusieurs algorithmes (régression logistique, forêt aléatoire, SVM, réseau de neurones) et on compare leurs performances.

In [ ]:
# Entraînement et comparaison de plusieurs modèles de machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(),
    'SVM': SVC(),
    'Neural Network': MLPClassifier(max_iter=500)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred)
    }
results_df = pd.DataFrame(results).T
print(results_df.sort_values('f1', ascending=False))

## 10. Sélection du meilleur modèle et analyse des résultats
On sélectionne le modèle ayant les meilleures performances globales et on analyse les erreurs (matrice de confusion, faux positifs/faux négatifs).

In [ ]:
# Analyse du meilleur modèle : matrice de confusion
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# On suppose que le meilleur modèle est celui avec le meilleur F1-score
top_model_name = results_df['f1'].idxmax()
top_model = models[top_model_name]
y_pred = top_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot()
plt.title(f'Matrice de confusion - {top_model_name}')
plt.show()

## 11. Sauvegarde du modèle entraîné
On sauvegarde le modèle pour l'intégrer ensuite dans le prototype WAF (Flask).

In [ ]:
# Sauvegarde du meilleur modèle entraîné
import joblib
joblib.dump(top_model, 'waf_model.joblib')
print(f"Modèle sauvegardé sous 'waf_model.joblib' ({top_model_name})")

## 12. Discussion, limites et perspectives
- Analyse des résultats obtenus
- Discussion sur les limites du modèle (généralisation, biais, attaques zero-day)
- Propositions d'améliorations (deep learning, détection d'anomalies, intégration SIEM)

Ce notebook fournit une base solide et structurée pour viser l'excellence dans votre projet de détection d'attaques web par machine learning.

In [ ]:
# Enregistrer les données nettoyées dans un nouveau fichier CSV
df.to_csv('donnees_nettoyees.csv', index=False)